# Do Bias Metrics Agree? #
**Benchmarking AIF360 and Fairlearn across Fairness Metrics and Mitigation Strategies**

**Main Notebook**  

**Datasets:** German Credit + COMPAS  
**Models:** Logistic Regression (baseline) + Random Forest  
**Framework:** AIF360 + Fairlearn  
*Victor Gabriel Dinu, Eddie Begovic | Summer Semester 2026*

---
## 0. Imports

In [1]:
# suppress aif360/tensorflow warnings
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd

# classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# preprocessing and pipeline utilities
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# evaluation metrics
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

import aif360
# StandardDataset is the base class — both datasets will be instances of this
from aif360.datasets import StandardDataset

SEED = 42
print('imports ok')

imports ok


---
## 1. Data Loading

In [2]:
# - German Credit - 

GERMAN_URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data'

# column names taken from the uci documentation for this dataset
GERMAN_COLS = [
    'status', 'month', 'credit_history', 'purpose', 'credit_amount',
    'savings', 'employment', 'investment_as_income_percentage', 'personal_status',
    'other_debtors', 'residence_since', 'property', 'age',
    'installment_plans', 'housing', 'number_of_credits',
    'skill_level', 'people_liable_for', 'telephone', 'foreign_worker', 'credit'
]

# read space-separated file with no header
df_g = pd.read_csv(GERMAN_URL, sep=' ', header=None, names=GERMAN_COLS)

# derive sex from personal_status codes (same mapping as aif360's GermanDataset source)
# A91/A93/A94 = male, A92/A95 = female
df_g['sex'] = df_g['personal_status'].map(
    {'A91': 'male', 'A93': 'male', 'A94': 'male', 'A92': 'female', 'A95': 'female'}
)

# convert labels to {0, 1} before passing to StandardDataset
# the raw file uses 1=good, 2=bad — aif360 would keep them as {1,2} otherwise
df_g['credit'] = (df_g['credit'] == 1).astype(float)  # good=1.0, bad=0.0

# wrap in StandardDataset: handles one-hot encoding of categoricals
# and maps protected attributes to binary privileged/unprivileged (1/0)
german = StandardDataset(
    df=df_g,
    label_name='credit',
    favorable_classes=[1],            # 1 = good credit = favorable outcome
    protected_attribute_names=['sex'],
    privileged_classes=[['male']],    # male = privileged group
    categorical_features=[
        'status', 'credit_history', 'purpose', 'savings', 'employment',
        'personal_status', 'other_debtors', 'property', 'installment_plans',
        'housing', 'skill_level', 'telephone', 'foreign_worker'
    ],
    features_to_drop=['personal_status'],  # drop original column — replaced by 'sex'
)

# - COMPAS -

# compas data comes packaged in aif360, so no download needed
COMPAS_PATH = os.path.join(
    os.path.dirname(aif360.__file__),
    'data', 'raw', 'compas', 'compas-scores-two-years.csv'
)

# read raw csv (index on 'id' to preserve row ids for for when we extract race later)
df_c = pd.read_csv(COMPAS_PATH, index_col='id', na_values=[])

# apply the same five filters that aif360's CompasDataset uses internally
df_c = df_c[
    (df_c['days_b_screening_arrest'] <= 30) &
    (df_c['days_b_screening_arrest'] >= -30) &
    (df_c['is_recid'] != -1) &
    (df_c['c_charge_degree'] != 'O') &
    (df_c['score_text'] != 'N/A')
]

# derived feature used by aif360's default compas preprocessing
df_c['length_of_stay'] = (
    pd.to_datetime(df_c['c_jail_out']) - pd.to_datetime(df_c['c_jail_in'])
).dt.days

compas = StandardDataset(
    df=df_c,
    label_name='two_year_recid',
    favorable_classes=[0],                        # 0 = no recidivism = favorable outcome
    protected_attribute_names=['sex', 'race'],
    privileged_classes=[['Female'], ['Caucasian']],
    categorical_features=['age_cat', 'c_charge_degree', 'c_charge_desc'],
    features_to_keep=['sex', 'age', 'age_cat', 'race', 'juv_fel_count',
                      'juv_misd_count', 'juv_other_count', 'priors_count',
                      'c_charge_degree', 'c_charge_desc', 'two_year_recid',
                      'length_of_stay'],
)

print(f'German Credit : {german.features.shape[0]} rows, {german.features.shape[1]} features')
print(f'  favorable_label={german.favorable_label} (good credit)')
print(f'COMPAS        : {compas.features.shape[0]} rows, {compas.features.shape[1]} features')
print(f'  favorable_label={compas.favorable_label} (no recidivism)')

German Credit : 1000 rows, 58 features
  favorable_label=1.0 (good credit)
COMPAS        : 6167 rows, 402 features
  favorable_label=0.0 (no recidivism)


Both datasets are now `StandardDataset` objects with the same interface. 

German Credit has 1000 rows and 58 features after one-hot encoding the categorical columns. `favorable_label=1.0` confirms the pre-binarisation worked — good credit maps to 1, bad credit to 0. 

COMPAS dropped 5 rows through its preprocessing filters (most likely instances of inconsistent screening dates or invalid charge types); this isn't suprising and matches what AIF360's built-in `CompasDataset` does. `favorable_label=0.0` is correct, so no recidivism is the favorable outcome, and since COMPAS labels were already `{0, 1}`, they stay as they are. 

---
## 2. EDA

In [3]:
# - German Credit -

print('=== GERMAN CREDIT ===')

# extract labels and protected attributes from the aif360 object
y_g   = german.labels.ravel()
sex_g = german.protected_attributes[:, 0]  # 1=male (privileged), 0=female
fav_g = german.favorable_label             # 1.0 = good credit

print(f'N={len(y_g)} | good credit: {(y_g == fav_g).mean():.1%} | male: {sex_g.mean():.1%}')

# good credit rate broken down by sex, shows pre-model disparity in the labels
for val, name in [(1, 'Male'), (0, 'Female')]:
    mask = sex_g == val
    print(f'  [{name}] n={mask.sum()} | good credit rate: {(y_g[mask] == fav_g).mean():.1%}')

# ─ COMPAS -

print()
print('=== COMPAS ===')

y_c    = compas.labels.ravel()
race_c = compas.protected_attributes[:, 1]  # 1=Caucasian (privileged), 0=other
fav_c  = compas.favorable_label             # 0.0 = no recidivism

print(f'N={len(y_c)} | no recid: {(y_c == fav_c).mean():.1%} | Caucasian: {race_c.mean():.1%}')

# no-recidivism rate broken down by race, shows pre-model disparity in the labels
for val, name in [(1, 'Caucasian'), (0, 'Non-Caucasian')]:
    mask = race_c == val
    print(f'  [{name}] n={mask.sum()} | no recid rate: {(y_c[mask] == fav_c).mean():.1%}')

=== GERMAN CREDIT ===
N=1000 | good credit: 70.0% | male: 69.0%
  [Male] n=690 | good credit rate: 72.3%
  [Female] n=310 | good credit rate: 64.8%

=== COMPAS ===
N=6167 | no recid: 54.5% | Caucasian: 34.1%
  [Caucasian] n=2100 | no recid rate: 60.9%
  [Non-Caucasian] n=4067 | no recid rate: 51.1%


**German Credit:** the dataset is imbalanced: 70% of cases are good credit. The key observation is the 7.5 point gap between male (72.3%) and female (64.8%) good credit rates. This disparity exists in the *ground truth labels*, so before any model is involved. Models trained on this data will consequently inherit at least some of it, which means fairness gaps that may be observed later will be partly a property of the data, not just the algorithm.

**COMPAS:** 54.5% of defendants did not reoffend within two years. Caucasians have a 60.9% no-recidivism rate versus 51.1% for non-Caucasians: roughly a 10 point gap. This is the baseline disparity that the fairness metrics in the next session will be measuring against.

---
## 3. Model Training

In [4]:
def train_eval(aif_ds, clf, label='', target_names=None):
    """
    Split → fit → evaluate.
    Returns (test_dataset, pipeline, y_pred, y_prob) for use in fairness cells later.
    """

    # 70/30 split using aif360's method — preserves StandardDataset format for later
    train, test = aif_ds.split([0.7], shuffle=True, seed=SEED)

    # scale features then fit — StandardScaler needed because LR is sensitive to scale
    pipe = make_pipeline(StandardScaler(), clf)
    pipe.fit(train.features, train.labels.ravel())

    y_pred = pipe.predict(test.features)
    # probability of the positive class — used for auc calculation
    y_prob = pipe.predict_proba(test.features)[:, 1]

    y_test = test.labels.ravel()
    print(label)
    print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}  |  AUC: {roc_auc_score(y_test, y_prob):.4f}')
    print(classification_report(y_test, y_pred, target_names=target_names))

    # return test set as aif360 object — needed for ClassificationMetric in the next session
    return test, pipe, y_pred, y_prob

In [5]:
test_lr_g, pipe_lr_g, y_pred_lr_g, y_prob_lr_g = train_eval(
    german,
    LogisticRegression(max_iter=1000, random_state=SEED),
    label='LR | German Credit',
    target_names=['Bad (0)', 'Good (1)']
)

LR | German Credit
Accuracy: 0.7500  |  AUC: 0.8044
              precision    recall  f1-score   support

     Bad (0)       0.61      0.51      0.56        92
    Good (1)       0.80      0.86      0.83       208

    accuracy                           0.75       300
   macro avg       0.70      0.68      0.69       300
weighted avg       0.74      0.75      0.74       300



Accuracy 0.75, AUC 0.80. Good performance on the majority class (good credit, F1 0.83), but the model struggles on bad credit (recall of 0.51 means it misses about half the actual bad credit cases). This is not unusual for an imbalanced dataset with no class weighting applied, as the model learns it can score well overall by favouring the majority class.

In [6]:
test_rf_g, pipe_rf_g, y_pred_rf_g, y_prob_rf_g = train_eval(
    german,
    RandomForestClassifier(n_estimators=100, max_depth=5, random_state=SEED, n_jobs=-1),
    label='RF | German Credit',
    target_names=['Bad (0)', 'Good (1)']
)

RF | German Credit
Accuracy: 0.7367  |  AUC: 0.8164
              precision    recall  f1-score   support

     Bad (0)       0.88      0.16      0.28        92
    Good (1)       0.73      0.99      0.84       208

    accuracy                           0.74       300
   macro avg       0.81      0.58      0.56       300
weighted avg       0.78      0.74      0.67       300



AUC improves slightly to 0.816 (better than LR's 0.804), meaning the RF's underlying probability scores rank cases better. However, recall on bad credit collapses to just 0.16 (it only catches 15 out of 92 actual bad credit cases). Also, the high precision (0.88) on bad credit is misleading: the model is so reluctant to predict bad credit that when it does, it's usually right, but it lets the vast majority of real bad cases through undetected. This threshold behaviour will likely affect the fairness metrics, by understating disparity gaps due to rarely predicting the unfavorable outcome. 

In [7]:
test_lr_c, pipe_lr_c, y_pred_lr_c, y_prob_lr_c = train_eval(
    compas,
    clf=LogisticRegression(max_iter=1000, random_state=SEED),
    label='LR | COMPAS',
    target_names=['No recid (0)', 'Recid (1)']
)

LR | COMPAS
Accuracy: 0.6710  |  AUC: 0.7110
              precision    recall  f1-score   support

No recid (0)       0.68      0.76      0.72      1023
   Recid (1)       0.65      0.56      0.60       828

    accuracy                           0.67      1851
   macro avg       0.67      0.66      0.66      1851
weighted avg       0.67      0.67      0.67      1851



Accuracy 0.671, AUC 0.711. Performance is more balanced across both classes than on German Credit (F1 0.72 vs 0.60). The lower overall AUC compared to German Credit (0.71 vs 0.80) reflects that recidivism is genuinely harder to predict; probably because financial features like income and employment are strong credit risk signals, whereas the features available in COMPAS carry much weaker signal about whether someone will reoffend.

In [8]:
test_rf_c, pipe_rf_c, y_pred_rf_c, y_prob_rf_c = train_eval(
    compas,
    clf=RandomForestClassifier(n_estimators=100, max_depth=5, random_state=SEED, n_jobs=-1),
    label='RF | COMPAS',
    target_names=['No recid (0)', 'Recid (1)']
)

RF | COMPAS
Accuracy: 0.6629  |  AUC: 0.7205
              precision    recall  f1-score   support

No recid (0)       0.65      0.86      0.74      1023
   Recid (1)       0.70      0.42      0.53       828

    accuracy                           0.66      1851
   macro avg       0.68      0.64      0.63      1851
weighted avg       0.67      0.66      0.64      1851



AUC improves to 0.721 versus 0.711 for LR: a gap of 0.010, comparable to the 0.012 gap seen on German Credit. The RF is picking up some non-linear structure in COMPAS, but the improvement is modest. Recall on recidivism drops to 0.42, consistent with the same threshold-skew seen on German Credit — the model defaults to predicting no recidivism far more often than it predicts recidivism.

In [9]:
# collect all four results into a single comparison table
summary = pd.DataFrame([
    {'Dataset': 'German Credit', 'Model': 'LR',
     'Accuracy': accuracy_score(test_lr_g.labels.ravel(), y_pred_lr_g),
     'AUC': roc_auc_score(test_lr_g.labels.ravel(), y_prob_lr_g)},
    {'Dataset': 'German Credit', 'Model': 'RF',
     'Accuracy': accuracy_score(test_rf_g.labels.ravel(), y_pred_rf_g),
     'AUC': roc_auc_score(test_rf_g.labels.ravel(), y_prob_rf_g)},
    {'Dataset': 'COMPAS',        'Model': 'LR',
     'Accuracy': accuracy_score(test_lr_c.labels.ravel(), y_pred_lr_c),
     'AUC': roc_auc_score(test_lr_c.labels.ravel(), y_prob_lr_c)},
    {'Dataset': 'COMPAS',        'Model': 'RF',
     'Accuracy': accuracy_score(test_rf_c.labels.ravel(), y_pred_rf_c),
     'AUC': roc_auc_score(test_rf_c.labels.ravel(), y_prob_rf_c)},
]).set_index(['Dataset', 'Model']).round(4)

display(summary)

Accuracy     AUC
Dataset       Model                  
German Credit LR       0.7500  0.8044
              RF       0.7367  0.8164
COMPAS        LR       0.6710  0.7110
              RF       0.6629  0.7205

RF consistently outperforms LR on AUC across both datasets, meaning its probability scores do a better job of ranking cases. It underperforms on accuracy and minority-class recall because it defaults to a threshold that skews heavily toward the majority class. When fairness metrics are added in the next session, we assume the two models will produce different selection rates across demographic groups; not because one is inherently fairer, but because of this difference in prediction behaviour. 